In [2]:
# Auto-switch to repo root so relative paths (configs/, scripts/, data/) work
from pathlib import Path
import os

_cwd = Path.cwd()
_root = _cwd
while _root != _root.parent:
    if (_root / "configs" / "config.yaml").exists():
        break
    _root = _root.parent
os.chdir(_root)
print(f"Repo root: {_root}")


Repo root: c:\Users\karki\Downloads\main_openworld\openworld


In [13]:
import json
from pathlib import Path

checkpoint = Path("outputs/checkpoints/full/epoch_3.pt")

with open("data/splits/val_test_image_ids.json") as f:
    test_ids = json.load(f)

with open("data/annotations/instances_val_test2017.stripped.json") as f:
    data = json.load(f)

image_names = {int(image["id"]): image["file_name"] for image in data["images"]}

missing_images = [
    image_id for image_id in test_ids
    if int(image_id) not in image_names
    or not (Path("data/images/val2017") / image_names[int(image_id)]).is_file()
]

missing_proposals = [
    image_id for image_id in test_ids
    if not (Path("outputs/proposals/val_test") / f"{int(image_id):012d}.npz").is_file()
]

print("Full checkpoint found:", checkpoint.is_file())
print("Test images missing:", len(missing_images))
print("Test proposal files missing:", len(missing_proposals))

Full checkpoint found: True
Test images missing: 0
Test proposal files missing: 0


In [8]:
%run scripts/extract_proposals.py --config configs/config.yaml --checkpoint outputs/checkpoints/full/epoch_3.pt --image-ids data/splits/val_test_image_ids.json --annotations data/annotations/instances_val_test2017.stripped.json --images-root data/images/val2017 --out-dir outputs/proposals/val_test --top-n 1000

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
processed 1500/1500
Saved 1500 proposal files -> outputs\proposals\val_test


In [15]:
%run scripts/run_clip.py --mode proposals --config configs/config.yaml --image-ids data/splits/val_test_image_ids.json --annotations data/annotations/instances_val_test2017.stripped.json --images-root data/images/val2017 --proposals-dir outputs/proposals/val_test --top-k 100 --alpha 0.5 --expansion 0.2 --out-dir outputs/clip/val_test_proposals

loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
mode=proposals images=1500 predictions=150000
saved -> outputs\clip\val_test_proposals\predictions.json


In [19]:
# 2. Confirm the files from your completed runs are present.
import numpy as np
from collections import defaultdict
from pycocotools.coco import COCO

pred_path = Path("outputs/clip/val_test_proposals/predictions.json")
ann_path = Path("data/annotations/instances_val_test2017.stripped.json")
ids_path = Path("data/splits/val_test_image_ids.json")
classes_path = Path("data/splits/selected_categories.json")
checkpoint_path = Path("outputs/checkpoints/full/epoch_3.pt")
for p in [pred_path, ann_path, ids_path, classes_path, checkpoint_path]:
    if not p.is_file():
        raise FileNotFoundError(f"Required file not found: {p}. Wait for the current CLIP run to finish if it is still running.")
with ids_path.open(encoding="utf-8") as f: image_ids = [int(i) for i in json.load(f)]
with classes_path.open(encoding="utf-8") as f: selected = json.load(f)
with pred_path.open(encoding="utf-8") as f: predictions = json.load(f)
coco = COCO(str(ann_path))
base = selected["base"]
novel = selected["novel"]
classes = base + novel
names = [c["name"] for c in classes]
cat_ids = [int(c["id"]) for c in classes]
if len(base) != 15 or len(novel) != 5 or len(set(cat_ids)) != 20:
    raise ValueError("Expected 15 base and 5 novel distinct categories.")
if not image_ids or not predictions:
    raise ValueError("The test split or the prediction file is empty.")
id_set = set(image_ids)
if len(id_set) != len(image_ids) or not id_set.issubset(coco.imgs):
    raise ValueError("The test image IDs are duplicated or missing from the annotations.")
invalid = [p for p in predictions if int(p["image_id"]) not in id_set or int(p["category_id"]) not in set(cat_ids)]
if invalid:
    raise ValueError(f"Found {len(invalid)} predictions with an unexpected image or class ID. Stop and ask for help.")
covered = len({int(p["image_id"]) for p in predictions})
print("Full checkpoint:", checkpoint_path)
print("Test images:", len(image_ids))
print("Predictions:", len(predictions))
print("Images with at least one saved prediction:", covered)
if covered != len(image_ids):
    raise ValueError("Some test images have no saved predictions. Check the completed CLIP run before evaluating.")
print("Files ready. Assumes these predictions were created with alpha 0.5 as instructed.")


loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Full checkpoint: outputs\checkpoints\full\epoch_3.pt
Test images: 1500
Predictions: 150000
Images with at least one saved prediction: 1500
Files ready. Assumes these predictions were created with alpha 0.5 as instructed.


In [20]:
# 3. Prepare ground truth and match predictions class by class.
# This checks class labels and includes test images with no detection for a class.
from collections import defaultdict

pred_by_image = defaultdict(list)
for p in predictions:
    box = np.asarray(p["bbox"], dtype=float)
    if box.shape != (4,) or not np.isfinite(box).all() or not np.isfinite(float(p["score"])):
        raise ValueError("Invalid prediction box or score.")
    pred_by_image[int(p["image_id"])].append(p)

gt_by_image = {}
for iid in image_ids:
    anns = coco.loadAnns(coco.getAnnIds(imgIds=[iid]))
    objects = []
    for a in anns:
        if int(a.get("iscrowd", 0)) or int(a["category_id"]) not in cat_ids:
            continue
        x, y, w, h = [float(v) for v in a["bbox"]]
        if w > 0 and h > 0:
            objects.append({"bbox": [x, y, x + w, y + h], "category_id": int(a["category_id"])})
    gt_by_image[iid] = objects

def overlaps(box, boxes):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    if not len(boxes): return np.zeros(0, dtype=float)
    box = np.asarray(box, dtype=float)
    lo = np.maximum(boxes[:, :2], box[:2])
    hi = np.minimum(boxes[:, 2:], box[2:])
    wh = np.maximum(hi - lo, 0)
    inter = wh[:, 0] * wh[:, 1]
    area_a = np.maximum(box[2] - box[0], 0) * np.maximum(box[3] - box[1], 0)
    wh_b = np.maximum(boxes[:, 2:] - boxes[:, :2], 0)
    union = area_a + wh_b[:, 0] * wh_b[:, 1] - inter
    return inter / np.maximum(union, 1e-8)

# Store each detection's match once; bootstrap later reweights whole images.
records = {cid: [] for cid in cat_ids}
gt_counts = {cid: np.zeros(len(image_ids), dtype=np.int64) for cid in cat_ids}
for idx, iid in enumerate(image_ids):
    groups = defaultdict(list)
    for g in gt_by_image[iid]: groups[g["category_id"]].append(g["bbox"])
    per_class_pred = defaultdict(list)
    for p in pred_by_image[iid]: per_class_pred[int(p["category_id"])].append(p)
    for cid in cat_ids:
        gboxes = groups[cid]
        gt_counts[cid][idx] = len(gboxes)
        used = set()
        for p in sorted(per_class_pred[cid], key=lambda d: -float(d["score"])):
            ious = overlaps(p["bbox"], gboxes)
            if used: ious[list(used)] = -1
            best = int(np.argmax(ious)) if len(ious) else -1
            # The corrected notebook 04 uses strictly greater than 0.5.
            hit = best >= 0 and ious[best] > 0.5
            if hit: used.add(best)
            records[cid].append((float(p["score"]), idx, int(hit)))
    if (idx + 1) % 250 == 0 or idx + 1 == len(image_ids):
        print(f"Checked {idx + 1}/{len(image_ids)} test images", flush=True)

arrays = {}
for cid in cat_ids:
    rec = sorted(records[cid], key=lambda r: -r[0])
    arrays[cid] = (np.asarray([r[1] for r in rec], dtype=np.int64),
                   np.asarray([r[2] for r in rec], dtype=float))

# Same 101-point interpolated AP used in the fixed notebook 04.
def class_ap(cid, multiplicity):
    n_gt = int(np.dot(gt_counts[cid], multiplicity))
    index, hits = arrays[cid]
    if n_gt == 0 or len(index) == 0: return 0.0
    weight = multiplicity[index]
    keep = weight > 0
    if not keep.any(): return 0.0
    weight, hits = weight[keep], hits[keep]
    tp = np.cumsum(weight * hits)
    fp = np.cumsum(weight * (1.0 - hits))
    recall = tp / n_gt
    precision = tp / np.maximum(tp + fp, 1e-12)
    envelope = np.maximum.accumulate(precision[::-1])[::-1]
    positions = np.searchsorted(recall, np.linspace(0, 1, 101), side="left")
    values = np.zeros(101)
    valid = positions < len(envelope)
    values[valid] = envelope[positions[valid]]
    return float(values.mean())

def group_scores(mult):
    per = [class_ap(cid, mult) for cid in cat_ids]
    return per, float(np.mean(per[:len(base)])), float(np.mean(per[len(base):])), float(np.mean(per))

per_class, base_ap, novel_ap, generalized_ap = group_scores(np.ones(len(image_ids), dtype=np.int64))
print(f"Base AP50:        {base_ap:.4f}")
print(f"Novel AP50:       {novel_ap:.4f}")
print(f"Generalized AP50: {generalized_ap:.4f}")


Checked 250/1500 test images
Checked 500/1500 test images
Checked 750/1500 test images
Checked 1000/1500 test images
Checked 1250/1500 test images
Checked 1500/1500 test images
Base AP50:        0.0314
Novel AP50:       0.0374
Generalized AP50: 0.0329


In [21]:
# 4. Image-level bootstrap confidence intervals.
# Images are sampled with replacement; repeated images retain their weight.
BOOTSTRAP_ROUNDS = 200
BOOTSTRAP_SEED = 42
rng = np.random.default_rng(BOOTSTRAP_SEED)
samples = np.empty((BOOTSTRAP_ROUNDS, 3), dtype=float)
for r in range(BOOTSTRAP_ROUNDS):
    drawn = rng.integers(0, len(image_ids), size=len(image_ids))
    mult = np.bincount(drawn, minlength=len(image_ids))
    _, b, n, g = group_scores(mult)
    samples[r] = [b, n, g]
    if (r + 1) % 25 == 0 or r + 1 == BOOTSTRAP_ROUNDS:
        print(f"Bootstrap: {r + 1}/{BOOTSTRAP_ROUNDS}", flush=True)

out_dir = Path("outputs/metrics")
out_dir.mkdir(parents=True, exist_ok=True)
intervals = {}
for i, key in enumerate(["base_AP50", "novel_AP50", "generalized_AP50"]):
    lo, hi = np.percentile(samples[:, i], [2.5, 97.5])
    intervals[key] = {"lower_95": float(lo), "upper_95": float(hi)}
    print(f"{key}: {intervals[key]['lower_95']:.4f} to {intervals[key]['upper_95']:.4f}")
report = {
    "split": "fixed val-test", "alpha": 0.5, "iou_threshold": 0.5,
    "ap_method": "custom 101-point per-class AP; non-crowd ground truth; strict IoU > 0.5; absent classes scored 0",
    "base_AP50": base_ap, "novel_AP50": novel_ap, "generalized_AP50": generalized_ap,
    "per_class_AP50": {name: ap for name, ap in zip(names, per_class)},
    "confidence_intervals_95": intervals, "bootstrap_rounds": BOOTSTRAP_ROUNDS,
    "bootstrap_seed": BOOTSTRAP_SEED, "num_images": len(image_ids), "num_predictions": len(predictions),
    "note": "Custom 20-class subset; not directly comparable to full OV-COCO. Alpha is assumed from the run that created the predictions."
}
report_path = out_dir / "openworld_eval.json"
with report_path.open("w", encoding="utf-8") as f: json.dump(report, f, indent=2)
print("Saved:", report_path)


Bootstrap: 25/200
Bootstrap: 50/200
Bootstrap: 75/200
Bootstrap: 100/200
Bootstrap: 125/200
Bootstrap: 150/200
Bootstrap: 175/200
Bootstrap: 200/200
base_AP50: 0.0288 to 0.0401
novel_AP50: 0.0282 to 0.0574
generalized_AP50: 0.0314 to 0.0429
Saved: outputs\metrics\openworld_eval.json


In [22]:
# 5. Display per-class results.
print(f"{'Class':14s} {'Split':7s} {'AP50':>7s}")
for i, (name, ap) in enumerate(zip(names, per_class)):
    split = "base" if i < len(base) else "novel"
    print(f"{name:14s} {split:7s} {ap:7.3f}")


Class          Split      AP50
person         base      0.054
car            base      0.035
chair          base      0.033
bottle         base      0.052
bicycle        base      0.035
motorcycle     base      0.022
truck          base      0.022
horse          base      0.060
sheep          base      0.020
bench          base      0.002
laptop         base      0.038
tv             base      0.017
book           base      0.012
backpack       base      0.020
bird           base      0.050
bus            novel     0.004
dog            novel     0.092
elephant       novel     0.086
umbrella       novel     0.004
keyboard       novel     0.002


In [ ]:
# 6. Confusion matrix and failure examples.
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

class_index = {cid: i for i, cid in enumerate(cat_ids)}
class_name = {int(c["id"]): c["name"] for c in classes}
matrix = np.zeros((len(classes), len(classes) + 1), dtype=int)
failures = []
for iid in image_ids:
    gts = gt_by_image[iid]
    candidates = sorted(pred_by_image[iid], key=lambda p: -float(p["score"]))
    available = set(range(len(gts)))
    matches = {}
    for p in candidates:
        if not available: break
        remaining = sorted(available)
        ious = overlaps(p["bbox"], [gts[j]["bbox"] for j in remaining])
        k = int(np.argmax(ious))
        if ious[k] > 0.5:
            j = remaining[k]
            available.remove(j)
            matches[j] = p
    for j, gt in enumerate(gts):
        p = matches.get(j)
        row = class_index[gt["category_id"]]
        col = class_index[int(p["category_id"])] if p else len(classes)
        matrix[row, col] += 1
        if col != row:
            failures.append({"image_id": iid, "gt": gt, "prediction": p})

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(matrix, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(names) + 1), names + ["missed"], rotation=90, fontsize=8)
ax.set_yticks(range(len(names)), names, fontsize=8)
ax.set_xlabel("Predicted class"); ax.set_ylabel("Actual class")
ax.set_title("Test-set object matches (IoU > 0.5)")
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
matrix_path = out_dir / "confusion_matrix.png"
fig.savefig(matrix_path, dpi=150)
plt.show()
print("Saved:", matrix_path)

# Save up to 12 example images. These are examples, not a ranked AP breakdown.
gallery = Path("outputs/analysis/gallery")
gallery.mkdir(parents=True, exist_ok=True)
saved = 0
for failure in failures:
    if saved >= 12: break
    iid, gt, pred = failure["image_id"], failure["gt"], failure["prediction"]
    image_path = Path("data/images/val2017") / coco.imgs[iid]["file_name"]
    if not image_path.is_file():
        raise FileNotFoundError(f"Missing test image: {image_path}")
    with Image.open(image_path) as img: image = img.convert("RGB")
    draw = ImageDraw.Draw(image)
    draw.rectangle(gt["bbox"], outline="lime", width=3)
    if pred: draw.rectangle(pred["bbox"], outline="red", width=3)
    actual = class_name[gt["category_id"]]
    guessed = class_name[int(pred["category_id"])] if pred else "missed"
    draw.text((5, 5), f"Actual: {actual} | Predicted: {guessed}", fill="yellow", stroke_width=2, stroke_fill="black")
    image.save(gallery / f"failure_{saved + 1:02d}.png")
    saved += 1
print(f"Saved {saved} example images to {gallery}")
print("Green = actual box; red = predicted box. A missed object has no red box.")


C:\Users\karki\AppData\Local\Temp\ipykernel_23636\77635911.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: outputs\metrics\confusion_matrix.png
Saved 12 example images to outputs\analysis\gallery
Green = actual box; red = predicted box. A missed object has no red box.


C:\Users\karki\AppData\Local\Temp\ipykernel_23636\77635911.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# 7. Optional detector-only COCO evaluation.
RUN_DETECTOR_BASELINE = True

if not RUN_DETECTOR_BASELINE:
    print("Detector-only baseline skipped. The open-world results above are already saved.")
else:
    import torch, yaml
    from torchvision.transforms import functional as TF
    from pycocotools.cocoeval import COCOeval
    from src.openworld.categories import label_map_for_base
    from src.openworld.detector import build_model

    with open("configs/config.yaml", encoding="utf-8") as f: cfg = yaml.safe_load(f)
    device = cfg["project"]["device"]
    label_map, _ = label_map_for_base("data/annotations/instances_train2017.stripped.json")
    inverse = {label: original for original, label in label_map.items()}
    model = build_model(num_classes=cfg["detector"]["num_foreground_classes"] + 1,
                        imagenet_backbone=cfg["detector"]["imagenet_backbone"],
                        trainable_backbone_layers=cfg["detector"]["trainable_backbone_layers"],
                        min_size=cfg["detector"]["min_size"], max_size=cfg["detector"]["max_size"]).to(device)
    # This is your own trusted checkpoint; it contains more than model weights.
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    detector_predictions = []
    with torch.inference_mode():
        for number, iid in enumerate(image_ids, start=1):
            path = Path("data/images/val2017") / coco.imgs[iid]["file_name"]
            with Image.open(path) as im: tensor = TF.to_tensor(im.convert("RGB")).to(device)
            output = model([tensor])[0]
            for box, score, label in zip(output["boxes"].cpu(), output["scores"].cpu(), output["labels"].cpu()):
                x1, y1, x2, y2 = map(float, box)
                detector_predictions.append({"image_id": iid, "category_id": int(inverse[int(label)]),
                                             "bbox": [x1, y1, x2-x1, y2-y1], "score": float(score)})
            if number == 1 or number % 25 == 0 or number == len(image_ids):
                print(f"Detector: {number}/{len(image_ids)} images", flush=True)
    det_path = Path("outputs/predictions/detector_val_test.json")
    det_path.parent.mkdir(parents=True, exist_ok=True)
    with det_path.open("w", encoding="utf-8") as f: json.dump(detector_predictions, f)
    if not detector_predictions:
        print("No detector predictions produced; COCO evaluation was not run.")
    else:
        # Some stripped JSONs omit info, which loadRes expects in current pycocotools.
        coco.dataset.setdefault("info", {})
        coco_dt = coco.loadRes(detector_predictions)
        evaluator = COCOeval(coco, coco_dt, "bbox")
        evaluator.params.imgIds = image_ids
        evaluator.params.catIds = sorted(int(c["id"]) for c in base)
        evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
        baseline = {"AP50": float(evaluator.stats[1]), "AP50_95": float(evaluator.stats[0]),
                    "num_images": len(image_ids), "num_predictions": len(detector_predictions),
                    "note": "Detector-only COCO evaluation on the 15 base categories."}
        base_path = out_dir / "detector_baseline_val_test.json"
        with base_path.open("w", encoding="utf-8") as f: json.dump(baseline, f, indent=2)
        print("Saved detector predictions:", det_path)
        print("Saved detector metrics:", base_path)


Detector: 1/1500 images
Detector: 25/1500 images
Detector: 50/1500 images
Detector: 75/1500 images
Detector: 100/1500 images
Detector: 125/1500 images
Detector: 150/1500 images
Detector: 175/1500 images
Detector: 200/1500 images
Detector: 225/1500 images
Detector: 250/1500 images


In [ ]:
## Finished

Main results: `outputs/metrics/openworld_eval.json`  
Confusion matrix: `outputs/metrics/confusion_matrix.png`  
Example images: `outputs/analysis/gallery/`

If a cell fails, stop there and share the **full output of that cell**. Do not rerun the expensive CLIP cell unless the saved prediction file is missing or incomplete.
